In [0]:
# 1. Parámetros (Widgets)
dbutils.widgets.text("storage_name", "")
dbutils.widgets.text("container_name", "bronze")

v_storage = dbutils.widgets.get("storage_name")
v_container = dbutils.widgets.get("container_name")

# 2. Configuración de la conexión
spark.conf.set(
    f"fs.azure.account.key.{v_storage}.dfs.core.windows.net",
    "AZURE_STORAGE_KEY_AQUI"
)


In [0]:
# Definimos la ruta de entrada
path_entrada = f"abfss://{v_container}@{v_storage}.dfs.core.windows.net"

# Leemos el JSON
df_productos = spark.read.json(f"{path_entrada}/productos.json")

print("Datos cargados en capa Bronze:")
display(df_productos)

Datos cargados en capa Bronze:


category,description,id,image,price,rating,title
men's clothing,"Your perfect pack for everyday use and walks in the forest. Stash your laptop (up to 15 inches) in the padded sleeve, your everyday",1,https://fakestoreapi.com/img/81fPKd-2AYL._AC_SL1500_t.png,109.95,"List(120, 3.9)","Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops"
men's clothing,"Slim-fitting style, contrast raglan long sleeve, three-button henley placket, light weight & soft fabric for breathable and comfortable wearing. And Solid stitched shirts with round neck made for durability and a great fit for casual fashion wear and diehard baseball fans. The Henley style round neckline includes a three-button placket.",2,https://fakestoreapi.com/img/71-3HjGNDUL._AC_SY879._SX._UX._SY._UY_t.png,22.3,"List(259, 4.1)",Mens Casual Premium Slim Fit T-Shirts
men's clothing,"great outerwear jackets for Spring/Autumn/Winter, suitable for many occasions, such as working, hiking, camping, mountain/rock climbing, cycling, traveling or other outdoors. Good gift choice for you or your family member. A warm hearted love to Father, husband or son in this thanksgiving or Christmas Day.",3,https://fakestoreapi.com/img/71li-ujtlUL._AC_UX679_t.png,55.99,"List(500, 4.7)",Mens Cotton Jacket
men's clothing,"The color could be slightly different between on the screen and in practice. / Please note that body builds vary by person, therefore, detailed size information should be reviewed below on the product description.",4,https://fakestoreapi.com/img/71YXzeOuslL._AC_UY879_t.png,15.99,"List(430, 2.1)",Mens Casual Slim Fit
jewelery,"From our Legends Collection, the Naga was inspired by the mythical water dragon that protects the ocean's pearl. Wear facing inward to be bestowed with love and abundance, or outward for protection.",5,https://fakestoreapi.com/img/71pWzhdJNwL._AC_UL640_QL65_ML3_t.png,695.0,"List(400, 4.6)",John Hardy Women's Legends Naga Gold & Silver Dragon Station Chain Bracelet
jewelery,Satisfaction Guaranteed. Return or exchange any order within 30 days.Designed and sold by Hafeez Center in the United States. Satisfaction Guaranteed. Return or exchange any order within 30 days.,6,https://fakestoreapi.com/img/61sbMiUnoGL._AC_UL640_QL65_ML3_t.png,168.0,"List(70, 3.9)",Solid Gold Petite Micropave
jewelery,"Classic Created Wedding Engagement Solitaire Diamond Promise Ring for Her. Gifts to spoil your love more for Engagement, Wedding, Anniversary, Valentine's Day...",7,https://fakestoreapi.com/img/71YAIFU48IL._AC_UL640_QL65_ML3_t.png,9.99,"List(400, 3.0)",White Gold Plated Princess
jewelery,Rose Gold Plated Double Flared Tunnel Plug Earrings. Made of 316L Stainless Steel,8,https://fakestoreapi.com/img/51UDEzMJVpL._AC_UL640_QL65_ML3_t.png,10.99,"List(100, 1.9)",Pierced Owl Rose Gold Plated Stainless Steel Double
electronics,"USB 3.0 and USB 2.0 Compatibility Fast data transfers Improve PC Performance High Capacity; Compatibility Formatted NTFS for Windows 10, Windows 8.1, Windows 7; Reformatting may be required for other operating systems; Compatibility may vary depending on user’s hardware configuration and operating system",9,https://fakestoreapi.com/img/61IBBVJvSDL._AC_SY879_t.png,64.0,"List(203, 3.3)",WD 2TB Elements Portable External Hard Drive - USB 3.0
electronics,"Easy upgrade for faster boot up, shutdown, application load and response (As compared to 5400 RPM SATA 2.5” hard drive; Based on published specifications and internal benchmarking tests using PCMark vantage scores) Boosts burst write performance, making it ideal for typical PC workloads The perfect balance of performance and reliability Read/write speeds of up to 535MB/s/450MB/s (Based on internal testing; Performance may vary depending upon drive capacity, host device, OS and application.)",10,https://fakestoreapi.com/img/61U7T1koQqL._AC_SX679_t.png,109.0,"List(470, 2.9)",SanDisk SSD PLUS 1TB Internal SSD - SATA III 6 Gb/s


In [0]:

from pyspark.sql.functions import col

# 1. Filtramos los registros que tienen ID (Válidos)
df_silver = df_productos.filter(col("id").isNotNull())

# 2. Filtramos los que NO tienen ID (Inválidos)
df_invalidos = df_productos.filter(col("id").isNull())

# 3. Guardamos los inválidos en una tabla Delta de LOGS
df_invalidos.write.format("delta").mode("append").saveAsTable("rejected_products_logs")

print(f"Registros Válidos: {df_silver.count()}")
print(f"Registros Rechazados: {df_invalidos.count()}")

Registros Válidos: 20
Registros Rechazados: 0


In [0]:

# Definimos la ruta de salida
path_salida = f"abfss://silver@{v_storage}.dfs.core.windows.net/productos_procesados"

# Guardamos como tabla Delta física
df_silver.write.format("delta").mode("overwrite").save(path_salida)

# Registramos la tabla en el catálogo para consultas SQL
df_silver.write.format("delta").mode("overwrite").saveAsTable("tabla_productos_final")

print("¡Pipeline completado! Los datos limpios están en la capa Silver/Gold.")

¡Pipeline completado! Los datos limpios están en la capa Silver/Gold.
